<a href="https://colab.research.google.com/github/rkuo2000/AI-exercise/blob/main/ResNet50_COVID_19_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ResNet50 COVID-19 Detection

## Section 1 — Imports & Environment Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split
# from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, f1_score, recall_score, precision_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, f1_score, recall_score, precision_score, brier_score_loss # updated 18 mar 2026
from sklearn.calibration import calibration_curve # updated 18 mar 2026

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms, models
from torchvision.transforms.functional import to_pil_image # # GRAD-CAM SETUP


from tqdm import tqdm
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Section 2 — Dataset Paths

In [ ]:
import kagglehub
database_path = kagglehub.dataset_download('tawsifurrahman/covid19-radiography-database')

import os
print(os.listdir(database_path))
dataset_path = os.path.join(database_path, "COVID-19_Radiography_Dataset")
print(os.listdir(dataset_path))

## Section 3 — Collect Image Paths

In [ ]:
covid_path = os.path.join(dataset_path,  "COVID/images")
normal_path = os.path.join(dataset_path, "Normal/images")

covid_images = [os.path.join(covid_path, img) for img in os.listdir(covid_path)]
normal_images = [os.path.join(normal_path, img) for img in os.listdir(normal_path)]

print("COVID images:", len(covid_images))
print("Normal images:", len(normal_images))

# Show Dataset Structure
print("\n\nDataset folder structure:")
for folder in os.listdir(dataset_path):
    path = os.path.join(dataset_path, folder)
    if os.path.isdir(path):
        print(folder, "->", len(os.listdir(os.path.join(path, "images"))), "images")

****So the dataset is imbalanced (Normal > COVID).****

## Section 4 — Create Labels

In [ ]:
data = []

# label encoding
# COVID = 1
# Normal = 0

for img in covid_images:
    data.append([img, 1])

for img in normal_images:
    data.append([img, 0])

# create dataframe
df = pd.DataFrame(data, columns=["image_path", "label"])

# shuffle dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# preview dataframe
print("Data Preview:")
print(df.head())
print("\nTotal images:", len(df))

# VERIFY IMAGE PATHS
missing = 0

for path in df["image_path"].head(100):
    if not os.path.exists(path):
        missing += 1

print("Missing files in first 100 samples:", missing)

## Section 5 — Class Distribution

In [ ]:
class_counts = df['label'].value_counts()

print("\nClass counts:")
print(class_counts)

print("\nClass percentages:")
print(df['label'].value_counts(normalize=True))

# nicer display
label_map = {0:"Normal", 1:"COVID"}
print("\nReadable class counts:")
print(df['label'].map(label_map).value_counts())


plt.figure(figsize=(6,4))

df['label'].map(label_map).value_counts().plot(kind='bar')

plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")

plt.show()

**Dataset Analysis**

Your dataset:
```
Class	Count	Percentage

Normal	10,192      73.8%
COVID	 3,616      26.2%
Total	13,808     100.0%
```
So the class ratio is roughly:
```
Normal: COVID
  2.8 : 1
```

This is a moderately imbalanced dataset.

## Section 6 — Visualize Sample Images

In [ ]:
fig, axes = plt.subplots(2,5, figsize=(15,6))

for i, ax in enumerate(axes.flat):

    idx = random.randint(0, len(df)-1)

    img_path = df.iloc[idx]['image_path']
    label = df.iloc[idx]['label']


    ## Even though X-rays are grayscale, converting to RGB is required for pretrained CNNs like:
    # ResNet
    # DenseNet
    # VGG
    img = Image.open(img_path).convert("RGB")

    ax.imshow(img)
    ax.set_title("COVID" if label==1 else "Normal")
    # Axis removal - Correct for image display
    ax.axis("off")

# plt.tight_layout() # OPTIONAL : before plt.show() so titles don't overlap.
plt.show()

# df.sample()

## Section 7 — Train / Validation / Test Split

We split the dataframe created earlier.

Typical ratio:

70% train

15% validation

15% test

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)

val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

#Verify Split Distribution
print("\nTrain distribution:")
print(train_df['label'].value_counts())

print("\nValidation distribution:")
print(val_df['label'].value_counts())

print("\nTest distribution:")
print(test_df['label'].value_counts())

print("\nTrain distribution (%)")
print(train_df['label'].value_counts(normalize=True))

**This produces:**
```
Dataset	        Percentage
Train	           70%
Validation         15%
Test               15%
```

**Results:**
```
Dataset        Images
Train           9665
Validation      2071
Test            2072
```

**Why stratify?**

Ensures class distribution/imbalance stays consistant across splits.

Because the splits are stratified:

- Validation metrics will be reliable

- Test metrics will represent real-world performance

Without stratification, the model could see:
```
Train: mostly Normal
Test: mostly COVID
```
which would completely break evaluation.

## Section 8 — Image Transformations

Medical imaging models almost always use 224×224 resolution because many pretrained models expect it.

We also apply data augmentation for training.

In [ ]:
IMG_SIZE = 224

# training transform
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# validation transform
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

**RandomHorizontalFlip**

Good because lungs are bilaterally symmetric.

**RandomRotation(10)**

Small rotations simulate:

- patient positioning

- scan misalignment

10 is a safe range.


------

**The values are ImageNet normalization values**

mean = [0.485, 0.456, 0.406]

std  = [0.229, 0.224, 0.225]

**Why these normalization values?**

They are commonly used when applying transfer learning with pretrained CNN models such as ResNet, VGG, and DenseNet, because these models were originally trained on the ImageNet dataset using the same normalization. Using the same statistics helps the pretrained weights generalize better.


## Section 9 — Custom PyTorch Dataset

We now create a dataset class that loads images dynamically.

In [ ]:
class XrayDataset(Dataset):
    # Resetting index is very important.
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    # This allows DataLoader to know dataset size.
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "image_path"]
        label = self.df.loc[idx, "label"]

        # Even though X-rays are grayscale, converting to RGB is necessary because pretrained CNNs expect 3 channels.
        image = Image.open(img_path).convert("RGB")

        # This allows:
            # train augmentations
            # validation preprocessing
            # using the same dataset class.
        if self.transform:
            image = self.transform(image)

        label = torch.tensor(label, dtype=torch.float32)

        return image, label

## Section 10 — Create Dataset Objects

In [ ]:
train_dataset = XrayDataset(train_df, train_transforms)
val_dataset = XrayDataset(val_df, val_transforms)
test_dataset = XrayDataset(test_df, val_transforms)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

## optional 3 lines
img, label = train_dataset[0]

print("\nImage shape:", img.shape)
print("Label:", label)

**Why use this?**

```
Dataset        Transform
Train          train_transforms (augmentation)
Validation     val_transforms (no augmentation)
Test           val_transforms (no augmentation)
```
Test data must not be augmented, so using val_transforms is correct.

This step/section converts pandas dataframe → PyTorch dataset.


So instead of:
```
image_path + label
```
you now have objects that can produce:
```
image tensor + label tensor
```
Which is required by DataLoader.

## Section 11 — DataLoaders

DataLoaders feed data into the model in batches.

In [ ]:
BATCH_SIZE = 32

# Adding, pin_memory=True.
# To speeds up GPU training. Recommended for CUDA environments (like Kaggle).

# Train data must shuffle to avoid learning order patterns.
# so, shuffle=True
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# Validation should be deterministic.
# so, shuffle=False
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Test set should never shuffle. So, shuffle=False
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# If the following code shows output:
# This confirms: transforms work, dataloader works, batch size correct

images, labels = next(iter(train_loader))

print("\nBatch image shape:", images.shape)
print("Batch label shape:", labels.shape)

For 224×224 X-rays with pretrained CNNs, typical batch sizes are:
```
Batch       Usage
16          very safe
32          ⭐ most common
64          sometimes too heavy
```
So 32 is a good default.

**Why `num_workers=2` Is Good for Kaggle**

Typical Kaggle environment:
```
workers     behavior
0           slow
2           ⭐ safe
4           sometimes unstable
8           rarely needed
```
So 2 is a good choice.

## Section 12 — Verify DataLoader

Always test the pipeline before training.

In [ ]:
images, labels = next(iter(train_loader))

# print("Image batch shape:", images.shape)
# print("Label batch shape:", labels.shape)

# images, labels = next(iter(train_loader))
print("Batch image shape:", images.shape)
print("Batch labels:", labels[:10])

# show first image
plt.imshow(images[0].permute(1,2,0))
plt.title(f"Label: {labels[0].item()}")
plt.axis("off")
plt.show()

Meaning:

32 images per batch

3 channels

224×224 resolution

**Why the warning??**

Why it happens:

We normalized the image earlier:

Normalize(
 mean=[0.485,0.456,0.406],
 std=[0.229,0.224,0.225]
)

Normalization changes pixel values from:

0 → 1

to roughly:

-2 → +2

But matplotlib.imshow() expects values in:

0 → 1

So matplotlib warns us.

## Section 13 — Load Pretrained Model (Transfer Learning)

We load a pretrained model and replace the classifier.

In [ ]:
# model = models.resnet50(pretrained=True)
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Freeze backbone layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer
num_features = model.fc.in_features

# model.fc = nn.Sequential(nn.Linear(num_features, 1)) # saves as "fc.0.weight" and "fc.0.bias"
model.fc = nn.Linear(model.fc.in_features, 1) # saves as "fc.weight" and "fc.bias"

# OPTIONAL
# Dropout helps especially when:
    # dataset imbalance exists
    # medical features are subtle
# But this is optional, not required.
# model.fc = nn.Sequential(
#     nn.Dropout(0.5),
#     nn.Linear(num_features, 1)
# )


# Our earlier output confirmed -    Using device: cuda
# So training will run on GPU.
model = model.to(device)

print(model)

Explanation:

Backbone (feature extractor) is frozen

Only final classifier is trained first

## Section 14 — Loss Function

Since this is binary classification, we use:

In [ ]:
normal_count = (df["label"]==0).sum() # 10192
covid_count  = (df["label"]==1).sum() # 3616

# COVID errors are penalized 2.8× more (10192 / 3616 = 2.8186). Model will care more about detecting COVID cases
pos_weight = torch.tensor([normal_count / covid_count]).to(device)

# pos_weight affects positive class (label = 1).

# Our label mapping:
    # Normal = 0
    # COVID = 1
# So the weight increases importance of COVID detection.
# This is perfect alignment.
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print("Positive class weight:", pos_weight)

### Interpretation of the Weight

Our weight:
```
2.8186
```
means during training:
```
COVID misclassification ≈ 2.8x more costly
```
This helps improve:

- Recall
- F1 score
- ROC AUC

Which are more important than accuracy in medical AI.

## Section 15 — Optimizer

We only train the classifier initially.

In [ ]:
optimizer = optim.Adam(
    model.fc.parameters(),
    lr=1e-4
)

Conceptually:
```
ResNet50 Backbone (Frozen)
        ↓
Feature Vector (2048)
        ↓
FC Layer (Trainable)
        ↓
Binary Prediction
```
This is standard transfer learning practice.

**Why Adam Is a Good Choice**

You used:
```
Adam
```
Advantages for medical datasets:
```
✔ faster convergence
✔ adaptive learning rate
✔ works well with small datasets
```
Many X-ray research papers also use Adam.

----------

Because we used:
```
model.fc.parameters()
```
our training is very efficient:

Only **2000** parameters train instead of **25 million** in ResNet50.


This means:
```
✔ faster training
✔ less overfitting
✔ lower GPU usage
```

----------

## Section 16 — Training Function

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):

    model.train()
    running_loss = 0

    for images, labels in tqdm(loader):

        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

## Section 17 — Validation Function

We compute predictions and metrics.

In [ ]:
def validate(model, loader, criterion):

    # This (model.eval())disables:
        # dropout
        # batchnorm updates
    # Exactly what we want during validation.
    model.eval()

    running_loss = 0
    all_preds = []
    all_labels = []

    # This (with torch.no_grad():) saves:
    #     memory
    #     computation
    # And prevents accidental gradient updates.
    with torch.no_grad():

        for images, labels in tqdm(loader):

            # Device Handling
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)# keeps shape consistent with model output.

            # Forward + Loss
            # Still using raw logits for loss.
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            preds = torch.sigmoid(outputs)

            # all_preds.extend(preds.cpu().numpy())
            all_preds.extend(preds.cpu().numpy().flatten())
            # all_labels.extend(labels.cpu().numpy())
            all_labels.extend(labels.cpu().numpy().flatten())


    # Returns:
    #     average validation loss
    #     predictions (probabilities)
    #     true labels
    return running_loss / len(loader), np.array(all_preds), np.array(all_labels)

## Section 18 — Training Loop

We train for multiple epochs.

In [ ]:
EPOCHS = 10  # number of epochs

# lists to store results
train_losses = []
val_losses = []
val_recalls = []
val_f1s = []
val_aucs = []

best_f1 = 0  # track best F1 score

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # Train for one epoch
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    # Validate
    val_loss, val_preds, val_labels = validate(
        model,
        val_loader,
        criterion
    )
# ************
    # Convert predictions
    # val_preds_binary = (val_preds > 0.5).astype(int)

    # Makes model more sensitive
    # redicts COVID more often
    val_preds_binary = (val_preds > 0.4).astype(int) # increases recall

# ************
    # Compute metrics
    recall = recall_score(val_labels, val_preds_binary)
    f1 = f1_score(val_labels, val_preds_binary)
    precision = precision_score(val_labels, val_preds_binary)
    auc = roc_auc_score(val_labels, val_preds)

    # store metrics AFTER computing them
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_recalls.append(recall)
    val_f1s.append(f1)
    val_aucs.append(auc)

    # print metrics
    print("Train Loss:", train_loss)
    print("Val Loss:", val_loss)
    print("Recall:", recall)
    print("Precision:", precision)
    print("F1 Score:", f1)
    print("ROC AUC:", auc)

# ****************
# Save the best model based on F1 score
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_model.pth")
        print("Best model saved!")

print("Training complete. Best model already saved as 'best_model.pth'.")

# ****************
# Specificity
tn, fp, fn, tp = confusion_matrix(val_labels, val_preds_binary).ravel()
specificity = tn / (tn + fp)

# Brier Score (use predicted probabilities, not binary)
brier = brier_score_loss(val_labels, val_preds)

# Calibration curve
prob_true, prob_pred = calibration_curve(val_labels, val_preds, n_bins=10)

# Print additional metrics
print("\n\n\nSpecificity:", specificity)
print("Brier Score:", brier)
# ****************

## Training Behavior (Very Important)

**Loss Trend**

Train Loss:
```c++
0.93 -> 0.55 (gradual decrease)
```
Val Loss:
```c++
0.89 -> 0.67 -> 0.59 (decreases, then stabilize)
```

***This means our model is learning properly.***




**Stability:**

- No sudden spikes

- Smooth learning across epochs

So: **Training process is stable and well-behaved**

## Model Performance

(focusing on the final epoch)
```
Recall:    0.906
Precision: 0.567
F1:        0.697
AUC:       0.923
```

### Interpretation


**Recall**
```c++
Recall = 0.85 - 0.91
```

Interpretation: ***Model detects most COVID cases (very high sensitivity)***


**Precision**

```c++
Precision = 0.45 - 0.61
```

 Meaning: ***Model produces some false positives***


**F1 Score**
```c++
F1 = 0.60 - 0.71
```
Interpretation: ***Balanced performance is moderate to good***

**ROC AUC**

```c++
AUC = 0.89 - 0.92
```

Interpretation: ***Model has strong class separation ability***


#### Specificity = 0.7547

This means: The model correctly identifies ~75.5% of normal (non-COVID) cases

Interpretation:

- Good (not perfect) ability to avoid false alarms

- Around 24.5% false positives still exist

- This is expected because you lowered threshold to 0.4, making the model more sensitive

Overall: ***Balanced but slightly biased toward predicting COVID (which is intentional)***

#### Brier Score = 0.1201

This measures probability quality (calibration)

Range:

- 0 = perfect

- 1 = worst

Interpretation:

- 0.12 is quite good

- Model probabilities are reasonably accurate

- Predictions are not just correct, but also well-calibrated

Example meaning: ***If model predicts 0.8 probability, it is often correct ~80% of the time***

## Overall Behaviour


Our model is:


**A high-recall classifier with good overall discrimination**



- Strong at detecting COVID

- Acceptable trade-off in precision

- Consistent performance across epochs















## Overfitting Analysis


**Compare losses**

```
Train Loss = 0.55 - 0.93

Val Loss = 0.59 - 0.89
```

Very close values: No major gap between training and validation


**Small signal:**
```
Val loss decreases until ~0.59

Then shows slight fluctuation in later epochs
```

This suggests:
```
Mild overfitting starting at later epochs
```

**Final verdict on overfitting:**

```
No strong overfitting, only slight overfitting at the end
```

- Model still generalizes well
- Overfitting is minimal and controlled

## Final Interpretation

Our model:
```
Is well-trained, stable, and suitable for medical detection tasks
```

**Strengths**

- High recall (critical for COVID detection)

- Strong AUC (good separation)

- Stable training

- Decent specificity (still handles normal cases reasonably well)

- Good probability calibration (trustworthy confidence scores)

- Handles imbalance effectively

---------

**Limitation**

## Lower precision → more false alarms

But:

## Acceptable in medical screening scenarios


---------

**Summary:**

The model learns effectively, achieves high sensitivity, maintains strong discrimination ability, and shows only mild overfitting.

## Section 18 (part 2) - LOADING SAVED MODEL


In [ ]:
# model = models.resnet50(pretrained=True)
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT) # updated 18 mar 2026
model.fc = nn.Linear(model.fc.in_features, 1)

model.load_state_dict(torch.load("best_model.pth"))
model.to(device)
model.eval()

## Section 19 — Visualization

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(range(1, len(train_losses)+1), train_losses, label="Train Loss")
plt.plot(range(1, len(val_losses)+1), val_losses, label="Validation Loss")

plt.legend()
plt.title("Training Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.show()

<!-- # Section 19 — Save Best Model

Always save the trained model. -->

## Section 20 — Test Set Evaluation

Now we evaluate the final model on unseen data.

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in tqdm(test_loader):

        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        outputs = model(images)

        preds = torch.sigmoid(outputs)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Binary predictions
# binary_preds = (all_preds > 0.5).astype(int)
binary_preds = (all_preds > 0.4).astype(int)  # changed to 0.4 to match val_preds

recall = recall_score(all_labels, binary_preds)
precision = precision_score(all_labels, binary_preds)
f1 = f1_score(all_labels, binary_preds)
auc = roc_auc_score(all_labels, all_preds)

print("TEST RESULTS")
print("Recall:", recall)
print("Precision:", precision)
print("F1 Score:", f1)
print("ROC AUC:", auc)

print("\nClassification Report:")
print(classification_report(all_labels, binary_preds, target_names=["Normal","COVID"]))

### Overall Performance

Our model achieves:

- **Accuracy = 0.83**

- **ROC AUC = 0.914**

This means:

- Strong overall classification ability

- Good separation between COVID and Normal cases

- Model generalizes well to unseen data

### PERFORMANCE COMPARISON

##### **COVID Class Performance (Important)**

- **Recall = 0.85**

- **Precision = 0.63**

- **F1 Score = 0.72**

Interpretation:

- Detects 85% of COVID cases

- About 15% of COVID cases are missed (false negatives)

- Predictions are fairly reliable (63% precision, so more false positives)


<!-- Compared to validation:

- Recall dropped (expected)

- Precision improved

- More balanced behavior overall -->



##### **Normal Class Performance**

- **Precision = 0.94**

- **Recall = 0.82**

- **F1 = 0.88**

Interpretation:

- Very strong at identifying normal cases

- Few false positives

- Stable and reliable predictions

### TRADE OFF


We now have two valid model behaviors:

**Threshold = 0.5** (all_preds > 0.5)

- More balanced

- Higher accuracy

- Lower COVID detection

**Threshold = 0.4** (all_preds > 0.4)

- High COVID detection (preferred in medical use)

- More false alarms

- Lower overall accuracy


<!-- **Generalization Check (VERY IMPORTANT)**

Compared with validation:

- Validation Recall was higher (~0.90+)

- Test Recall = 0.73

This indicates:

- Some drop in sensitivity on unseen data

- Mild overfitting to validation set

BUT:

- ROC AUC still high (0.91)

- Accuracy strong (0.88)

So: ***No major overfitting — just realistic performance drop*** -->

### For our project (COVID detection):

**Threshold = 0.4 is better**

Because:

- Missing COVID (false negative) is more dangerous than false alarms

- High recall (0.85) is more valuable

Lowering the decision threshold to 0.4 improves COVID detection significantly by increasing recall, at the cost of reduced precision and overall accuracy, making it more suitable for safety-critical medical screening.

## Section 21 — Confusion Matrix

This shows true/false predictions.

In [ ]:
# ===============================
# CONFUSION MATRIX
# ===============================

cm = confusion_matrix(all_labels, binary_preds)

# print matrix values
print("\nConfusion Matrix:")
print(cm)

TN, FP, FN, TP = cm.ravel()

print("\nDetailed Breakdown:")
print(f"True Negatives (Normal predicted Normal): {TN}")
print(f"False Positives (Normal predicted COVID): {FP}")
print(f"False Negatives (COVID predicted Normal): {FN}")
print(f"True Positives (COVID predicted COVID): {TP}")

# plot confusion matrix
plt.figure(figsize=(6,5))

plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()

classes = ["Normal","COVID"]

plt.xticks(range(2), classes)
plt.yticks(range(2), classes)

plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j], ha="center", va="center")

plt.show()

## **Medical Interpretation (Very Important)**

- High detection of COVID → 460 / 543

- Only 83 missed cases

- Strong sensitivity (recall ≈ 0.85)

**This is critical for screening tasks**

**Trade-off**

- 275 false positives

- Some normal patients flagged as COVID

Leads to:

- Extra testing

- Mild inconvenience

- But safer than missing real cases

## Section 22 — ROC Curve

Common metric in medical research.

In [ ]:
fpr, tpr, thresholds = roc_curve(all_labels, all_preds)

plt.figure(figsize=(6,5))

plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0,1],[0,1],"--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.show()

What it shows:

- X-axis = False Positive Rate (FPR)

- Y-axis = True Positive Rate (Recall)

Our result:

- AUC ≈ 0.91



This means:

**The model can separate COVID vs Normal very well**

**AUC > 0.9 = excellent discrimination**

Interpretation: ***If we randomly pick one COVID and one Normal image,
the model will rank the COVID case higher ~91% of the time***

## Section 23 — Precision-Recall Curve

Useful for imbalanced datasets.

In [ ]:
from sklearn.metrics import precision_recall_curve

precision_vals, recall_vals, _ = precision_recall_curve(all_labels, all_preds)

plt.figure(figsize=(6,5))

plt.plot(recall_vals, precision_vals)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")

plt.show()

For our case:

- Dataset is imbalanced (Normal > COVID)

- PR curve is more informative than ROC


Meaning:

- Shows how precision changes as recall increases

- Helps us choose the best threshold

**Our Model Behavior (based on earlier results)**

- As recall increases → precision decreases

This confirms your earlier observation (Lower threshold (0.4)):

- Higher recall

- Lower precision

## Section 24 — Grad-CAM (Explainability)

This visualizes where the CNN is looking in the lungs.

Grad-CAM is widely used in medical papers.

In [ ]:
target_layer = model.layer4[-1]

gradients = None
activations = None

def save_gradient(module, grad_input, grad_output):
    global gradients
    gradients = grad_output[0]

def save_activation(module, input, output):
    global activations
    activations = output

# register hooks
target_layer.register_forward_hook(save_activation)
# target_layer.register_backward_hook(save_gradient) # appratently depriciated
target_layer.register_full_backward_hook(save_gradient)

print("Grad-CAM hooks registered successfully")

## Section 25 — Grad-CAM Visualization Function

In [ ]:
def generate_gradcam(image_tensor):

    model.eval()

    image_tensor = image_tensor.unsqueeze(0).to(device)

    output = model(image_tensor)
    model.zero_grad()
    output.backward()

    pooled_gradients = torch.mean(gradients, dim=[0,2,3])
    activation = activations.squeeze(0)

    for i in range(len(pooled_gradients)):
        activation[i,:,:] *= pooled_gradients[i]

    heatmap = torch.mean(activation, dim=0).cpu().detach().numpy()
    heatmap = np.maximum(heatmap, 0)
    # heatmap /= np.max(heatmap) # If max = 0 → division error
    heatmap = heatmap / (np.max(heatmap) + 1e-8)

    return heatmap

**What This Does**

Grad-CAM will:

- Highlight where the model is looking

- Show important regions in X-ray images

Help verify:

- Is the model focusing on lungs?

- Or random noise?

## Section 26 — Show Grad-CAM Example

In [ ]:
import cv2

sample_img, label = test_dataset[1003]

heatmap = generate_gradcam(sample_img)

# debug information
print("Heatmap min:", heatmap.min())
print("Heatmap max:", heatmap.max())
print("Heatmap shape:", heatmap.shape)

# -------------------------------
# Denormalize image
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

img = sample_img.cpu().clone()

for t, m, s in zip(img, mean, std):
    t.mul_(s).add_(m)

img = img.permute(1,2,0).numpy()
img = np.clip(img, 0, 1)

# -------------------------------
# Resize heatmap to match image
heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

# -------------------------------
# Plot
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Original X-ray")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(img)
plt.imshow(heatmap, cmap="jet", alpha=0.5)
plt.title("Grad-CAM")
plt.axis("off")

plt.show()

## Section 27 — Grad-CAM Interpretation

**Visual Explanation of Model Decisions**

Grad-CAM was used to visualize the regions of the chest X-ray images that the model focuses on when making predictions. The heatmap highlights the most important areas contributing to the classification.

**Visual Explanation of Model Decisions**

Grad-CAM was used to visualize the regions of the chest X-ray images that the model focuses on when making predictions. The heatmap highlights the most important areas contributing to the classification.

**Interpretation**

This indicates that the model is:

- Learning meaningful medical features

- Not relying on random noise or artifacts

- Making decisions based on anatomically relevant structures

**Reliability Insight**

- The Grad-CAM visualization improves trust in the model by showing that:

- Predictions are explainable

- The model behavior aligns with medical expectations


**Reliability Insight**

- The Grad-CAM visualization improves trust in the model by showing that:

- Predictions are explainable

- The model behavior aligns with medical expectations

**Final Conclusion**

Grad-CAM results suggest that the model makes decisions based on relevant lung features, supporting its reliability for COVID-19 detection tasks.

## CONCLUSION


Our Final Metrics (from Section 20)

```
Using threshold = 0.4:

Recall = 0.847
👉 Model detects ~85% of COVID cases (very important medically)

Precision = 0.626
👉 Some false alarms (normal predicted as COVID)

F1 Score = 0.720
👉 Balanced performance

ROC AUC = 0.914
👉 Strong overall classification ability

Accuracy = 0.83
```


**This means**

Our model is:

- Good at detecting COVID (high recall)

- Sometimes over-predicts COVID (lower precision)

- Overall strong model (AUC > 0.9)


**In medical problems:**

- Recall > Precision is preferred

Because:
```
Missing a COVID case (false negative) = dangerous

False alarm (false positive) = safer
```

Our model is correctly tuned for this priority



**The final model achieved strong performance on the test set, with high recall for COVID detection and an AUC above 0.9, indicating good discriminative ability. The model prioritizes detecting positive cases, which is suitable for medical screening tasks.**

